# 02 - Plot Basic Quantities

Build a small event table from Level2 Physics frames, then plot hit DOM count, total charge, and any simple reconstruction quantities found in the file.


In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import pandas as pd

from icetray_tutorial.paths import EXPERIMENT, SIMULATION
from icetray_tutorial.frames import event_header_dict, iter_frames, stop_name
from icetray_tutorial.filters import passed_filter_names
from icetray_tutorial.pulses import summarize_pulses

In [ ]:
def particle_fields(frame, candidate_keys=('SplineMPE', 'MPEFit', 'LineFit', 'OnlineL2_SplineMPE')):
    for key in candidate_keys:
        if key in frame:
            particle = frame[key]
            return {
                'reco_key': key,
                'zenith_rad': float(getattr(particle.dir, 'zenith', float('nan'))),
                'azimuth_rad': float(getattr(particle.dir, 'azimuth', float('nan'))),
                'energy_GeV': float(getattr(particle, 'energy', float('nan'))),
            }
    return {'reco_key': None, 'zenith_rad': float('nan'), 'azimuth_rad': float('nan'), 'energy_GeV': float('nan')}

def build_event_table(i3_file, max_physics_frames=500):
    rows = []
    seen = 0
    for frame in iter_frames(i3_file):
        if stop_name(frame) != 'Physics':
            continue
        seen += 1
        row = event_header_dict(frame)
        pulse_summary = summarize_pulses(frame)
        if pulse_summary:
            row.update({'pulse_key': pulse_summary.pulse_key, 'hit_doms': pulse_summary.hit_doms, 'pulses': pulse_summary.pulses, 'total_charge': pulse_summary.total_charge})
        row.update(particle_fields(frame))
        row['passed_filters'] = ','.join(passed_filter_names(frame)[:10])
        rows.append(row)
        if seen >= max_physics_frames:
            break
    return pd.DataFrame(rows)

In [ ]:
exp_events = build_event_table(EXPERIMENT.event_file, max_physics_frames=500)
exp_events.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
exp_events['hit_doms'].dropna().hist(ax=axes[0], bins=40)
axes[0].set_xlabel('hit DOMs')
axes[0].set_ylabel('events')
exp_events['total_charge'].dropna().hist(ax=axes[1], bins=40)
axes[1].set_xlabel('total charge [PE]')
axes[1].set_ylabel('events')
plt.tight_layout()

In [ ]:
sim_events = build_event_table(SIMULATION.event_file, max_physics_frames=500)
ax = exp_events['hit_doms'].dropna().plot.hist(bins=40, alpha=0.5, label='exp')
sim_events['hit_doms'].dropna().plot.hist(ax=ax, bins=40, alpha=0.5, label='sim')
ax.set_xlabel('hit DOMs')
ax.legend()

## Practice

1. Plot reconstructed zenith in degrees.
2. Count named filters in the first 500 events.
3. Try a different pulse key after inspecting the frame.
